# fileCompare

## Imports

In [5]:
import pandas as pd
import sys,os,inspect
from operator import itemgetter
import csv
import json
# sys.path.insert(0, '/mnt/c/users/cojoe/Python-Stuff')
# import JCLib
import threading
#import PySimpleGUI as sg
import pyglet,tkinter
from pyglet import font
# import OpenGL
# from OpenGL import GLU
font.add_file('/etc/fonts/fonts/CENTAUR.TTF')
#D7BDE2 
##  Variables related to system info or imports
platform = sys.platform 
import time

colorPairs = [["#C39BD3","#D7BDE2"],["#D6EAF8","#85C1E9"],["#b3f0ff","#33d6ff"],["#D5F5E3","#A3E4D7"],["#FCF3CF","#F7DC6F"]]

In [4]:
! pip install pyglet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.1/962.1 kB 11.5 MB/s eta 0:00:00


## Functions

In [ ]:
def exceptionLog(exception,funCall):
  exception_message = str(exception)
  exception_type, exception_object, exception_traceback = sys.exc_info()
  filename = os.path.split(exception_traceback.tb_frame.f_code.co_filename)[1]
  print(f"{exception_message} {exception_type} {funCall}, Line {exception_traceback.tb_lineno}")
    
#####################################################################################

def find_delimiter(filename):
    sniffer = csv.Sniffer()
    with open(filename) as fp:
        delimiter = sniffer.sniff(fp.read(5000)).delimiter
    return delimiter

#####################################################

def fileAnalysis(file):
    delim = find_delimiter(file)    
    columns = {}
    fin = open(file,"r",encoding="latin")    
    lines = fin.readlines()
    for line in lines:
        spl = line.split(delim)
        nc = len(spl)
        if nc in columns:
            columns[nc]+=1
        else:
            columns[nc]=1
            
    return len(lines),delim,columns

########################################################################################
def analyzeNFields(file,delim=",",encoding="utf-8"):
    fin=open(file,"r",encoding=encoding)
    nlines=0
    histNF={}
    for line in fin:
        nlines+=1
        nn=len(line.split("\t"))
        if nlines > 1:
            if nn not in histNF:
                histNF[nn]=0
            histNF[nn]+=1
        else:
            headerLength=nn
            
    return headerLength,histNF

########################################################################################

def getFile(how,file,WindowP,quoting,header=0):
    if how == "Local":
        print("QUTING 2",quoting)
        if file[-3:].lower() == "tsv":
            try: 
                delim = "\t"
                print("try reading TSV file:",file)
                df=pd.read_csv(file,encoding="latin",delimiter=delim,header=header,quoting=quoting)
                headerLength,histNF=analyzeNFields(file,delim=delim)            
            except Exception as err:
                print("failed, now try latin for  reading TSV file:",file)
                
                df=pd.read_csv(file,encoding="latin",delimiter=delim,quoting=quoting)
                headerLength,histNF=analyzeNFields(file,delim=delim,encoding="latin")    
        elif file[-4:].lower() == "xlsx":
            df=pd.read_excel(file,engine="openpyxl",header=header)
        elif file[-3:].lower() == "xls":
            df=pd.read_excel(file,header=header)
        else:
            try:
                df=pd.read_csv(file)
                headerLength,histNF=analyzeNFields(file)
            except Exception as err:
                print("Error, trying with encoding=latin")
                df=pd.read_csv(file,encoding="latin")
                headerLength,histNF=analyzeNFields(file,encoding="latin")
    elif how == "Fetch":
        print("GETTING FETCH FILE")
   #     file=values["-WEB-"]
 #       getPrevFiles(2,file)

        WindowP["-PINFO-"].update(f"START reading WEB File:{file}:")
        df=pd.read_csv(f"{file}?$limit=999999999")
        print("GOT ME A FETCH FILE")
        WindowP["-PINFO-"].update(f"FINISHED reading WEB File:{file}:")
        print("DONE FETCHING ")
    return df

##############################################################

def setRowColors(lst,col1,col2,colsp,header):
    count=0
    colors = {}
    # print("COLORS ",col2,col2)
    # print("LIST ",lst)
    nrec = header.index("% Missing")
    for vals in lst:
        key = vals[0]
        if count%2 == 0:
            colors[key] = col1
        else:
            colors[key] = col2
        if vals[nrec] > 99.0:
            colors[key] = colsp
        count+=1
    colTab = []
    for key,colr in colors.items():
        colTab.append(colr)
    rowNums = [num for num in range(0,len(colTab)+1)]
    colText = ["black"]*len(colTab)
    colrw = list(zip(rowNums,colTab))
    print("COLORS ",colrw)
    return colrw

###########################################################

def sortTable(row,stats,colSortState,table,event,header):
    
        e = table.user_bind_event 
        region = table.Widget.identify('region', e.x, e.y)
        if region == 'heading':
            row = 0
        elif region == 'cell':
            row = int(table.Widget.identify_row(e.y))
   
        if row == 0:
            colClicked = int(table.Widget.identify_column(e.x)[1:])
            statClicked = header[colClicked-1].strip()
            colSortState[statClicked]*=-1
            if colSortState[statClicked] == -1:
                sortAsc=False
            else:
                sortAsc=True
            if colClicked > 1:  # user number sort
                statsS = dict(sorted(stats.items(), key=lambda x: x[1][statClicked],reverse=sortAsc))
            else:
                statsS = dict(sorted(stats.items(), key=lambda x: x[0],reverse=sortAsc))


            statsVals=[]
            for col in statsS:
                 vals=[]
                 vals.append(col)
                 for k in header[1:]:
                    vals.append(statsS[col][k.strip()])
                 statsVals.append(vals)
           # slen= len(statsVals)


#             window['-TABLE-'].update(values=statsVals,row_colors=colorsTable)
        return statsVals,colSortState

#######################################################################

def dfAnalyze(df):
    stats = {}
    
    for col in df.columns:
        typs = df[col].apply(type).value_counts().to_dict()
        stats[col] = {}
        if str in typs:
           stats[col]["string"] = typs[str]
        else:
           stats[col]["string"] = 0

        if int in typs:
           stats[col]["integer"] = typs[int]
        else:
           stats[col]["integer"] = 0

        if float in typs:
           stats[col]["float"] = typs[float]
        else:
           stats[col]["float"] = 0

        if bool in typs:
           stats[col]["boolean"] = typs[bool]
        else:
           stats[col]["boolean"] = 0

        stats[col]["Missing"] = df[col].isna().sum()
        stats[col]["% Missing"] = round(df[col].isna().sum()/df.shape[0]*100,1)
    print("FINISHED ANALYZING")
    return stats

#############################################  

def getValues(stats,header):
    print("GEtting Vals")
    statsVals=[]
    for col in sorted(stats.keys()):
         vals=[]
         vals.append(col)
         for k in header[1:]:
            vals.append(stats[col][k.strip()])
         statsVals.append(vals)
    return statsVals   

############################################################

def getFilesClicked(event,values,window,quoting):
        print("Get Files")
        print("QUOTING ",quoting)
        print(event)
        print(type(values),type(values["-WEB1-"]))
        print(len(values["-WEB1-"]))
        print(values)
        if event == "-FILE1-" and len(values["-FILE1-"]) > 0:
            file = values["-FILE1-"]
            how="Local"
        elif event == "Fetch" and len(values["-WEB1-"]) > 0:
            file = values["-WEB1-"]
            how="Fetch"
            
        print("Getitng FIle ",file)
        df = getFile(how,file,window,quoting)
        print("ANALYZING")
        stats = dfAnalyze(df)

        return df,stats,file

############################################################

def getRowClicked(table,columns):
    col=""
    e = table.user_bind_event 
    region = table.Widget.identify('region', e.x, e.y)
    if region == 'heading':
        row = 0
    elif region == 'cell':
        row = int(table.Widget.identify_row(e.y))  
        col = columns[row-1][0]
    return row,col    

#####################################################################

def showDFUN(col,unq,windowParent,colr1,colr2):
    totrows = sum(unq.tolist())
    valuesUNQ = list(zip(unq.index.tolist(),unq.tolist()))
    hUNQ = []
    hUNQ.append("Values")
    hUNQ.append("Count")
    ff=windowParent.metadata[2]
    
    sortState = {}
    for val in hUNQ:
        sortState[val]=-1
    layout2 = [    
                   [sg.Text(f"Showing unique Values for Column",font='Courier 10 bold '),sg.Text(f" {col}",text_color="red",font='Courier 10 bold ')],
                   [sg.Text(f"{len(valuesUNQ)} Unique Values Found",font='Courier 10 bold ')],
                   [sg.Text(f"{totrows} Non-Missing Values Found",font='Courier 10 bold ')],
        
                   [sg.Button('Quit')],
            
                   [sg.Button('Write Unique'),
                    sg.Table(values=valuesUNQ,
                       background_color=colr1,vertical_scroll_only=False,col_widths=60,font='Courier 10 bold ' ,
                       auto_size_columns=True,enable_events=True,def_col_width=25,text_color="black",
                       justification='right',alternating_row_color=colr2,
                       key='-TABLE-', headings = hUNQ,metadata=sortState)]
            ]
    window2 = sg.Window(f"Unique", layout2,finalize=True,debugger_enabled=True,resizable=True,metadata=[valuesUNQ,col,ff])
    
    table = window2['-TABLE-']
    table.bind('<Button-1>', "Click")
    
    return window2,table,valuesUNQ,hUNQ


###########################################################

def sortUniqe(table,window,dataStore,headerStore):
    try:
        e = table.user_bind_event
        region = table.Widget.identify('region', e.x, e.y)
        sortAsc = {}
        sortAsc[1] =False
        sortAsc[-1]=True
      
        if region == 'heading':
            values = dataStore[table]
           
            header = headerStore[table]
           
            column = int(table.Widget.identify_column(e.x)[1:])
            col=header[column-1]
          
            sortState = table.metadata
            sortState[col]*=-1
            table.metadata = sortState
          
            values = sorted(values, key=lambda element: (element[column-1]),reverse=sortAsc[sortState[col]]) 
          
            window["-TABLE-"].update(values=values)
            dataStore[table] = values
    except Exception as err:
        exceptionLog(err,inspect.currentframe().f_code.co_name)
                    
 #######################################################################################       
        
def dfStringAnal(df,windowP):
    stats = {}
    nstats = {}
    ndates={}
    nums = ["int64","float64"]
    columnsNoDate=[]
    try: 
        for col in df: 
            if df[col].dtypes == "object":
                try : 
                    stats[col] = {}
                    
                    stats[col]["digits"] = -1
                    stats[col]["non-digits"] = -1
                    stats[col]["numeric"] = -1
                    stats[col]["word"] = -1
        #            stats[col]["non-word"] = df["B1_PER_ID1"].str.contains("\S").sum()
                    stats[col]["non-word"] = -1
                    stats[col]["white-spc"] = -1
                    stats[col]["_"] = -1                     
                    stats[col]["-"] = -1
                    stats[col]["#"] = -1
                    stats[col]["missing"] = -1

                    columnsNoDate.append(col)
                    typs = df[col].apply(type).value_counts().to_dict()
                    
                    if str in typs:
                       stats[col]["string"] = typs[str]
                    else:
                       stats[col]["string"] = 0

                    if int in typs:
                       stats[col]["integer"] = typs[int]
                    else:
                       stats[col]["integer"] = 0

                    if float in typs:
                       stats[col]["float"] = typs[float]
                    else:
                       stats[col]["float"] = 0
                    
                    if bool in typs:
                       stats[col]["boolean"] = typs[bool]
                    else:
                       stats[col]["boolean"] = 0
                    
                    stats[col]["missing"] = df[col].isna().sum() 
                    stats[col]["% Missing"] = df[col].isna().sum()/df.shape[0]*100 
                    stats[col]["% Missing"] = float(f"{stats[col]['% Missing']:4.1f}")

                    stats[col]["digits"] = df[col].str.contains("\d").sum()
                    stats[col]["non-digits"] = df[col].str.contains("\D").sum()
                    stats[col]["numeric"] = df[col].str.replace(".","",1).str.isdecimal().sum()


                    stats[col]["word"] = df[col].str.contains("\w").sum()
        #            stats[col]["non-word"] = df["B1_PER_ID1"].str.contains("\S").sum()
                    stats[col]["non-word"] = df[col].str.contains("[^a-zA-Z0-9_ \-]").sum() 
                    stats[col]["white-spc"] = df[col].str.contains("\s").sum()   

                    stats[col]["_"] = df[col].str.contains("_").sum()                        
                    stats[col]["-"] = df[col].str.contains("-").sum()   
                    stats[col]["#"] = df[col].str.contains("#").sum() 
                except Exception as errs:
                    print(f"{col} Object error {errs}")
                    windowP["-PINFO-"].update(f"{col} Object error {errs}\n",append=True)
                    
            elif df[col].dtypes in nums:
                try: 
                    columnsNoDate.append(col)            
                    amin,amax,amen,astd = df[col].agg(["min","max","mean","std"])
                    nstats[col]={}
                    nstats[col]["Min"] = amin
                    nstats[col]["Max"] = amax
                    nstats[col]["Mean"] = amen
                    nstats[col]["Std"] = astd
                    nstats[col]["Missing"] =df[col].isna().sum()
                    nstats[col]["% Missing"] = df[col].isna().sum()/df.shape[0]*100 
                    nstats[col]["% Missing"] = float(f"{nstats[col]['% Missing']:4.1f}")
                    
                except Exception as errs:
                    print(f"{col} Number error {errs}")
   #                 windowP["-PINFO-"].update(f"{col} Number error {errs}\n",append=True)
                    
                    
            else:
                ndates[col]={}
                ndates[col]["Start"] = df[col].min()
                ndates[col]["End"] = df[col].max()
                ndates[col]["# Unique Dates"] = df[col].nunique()
    except Exception as err:
        print("dfStringAnal Error")
        print(err)

            
    
            
    nrecs=0
    return stats,nstats,ndates,columnsNoDate




def showDetailedAnalysis(dfO,windowP,file,idcol=""):
    
    global stats,nstats,statsVals,header_list
    df = dfO.copy()
    regex = {}
    regex["digits"] = "\d"
    regex["non-digits"] = "\D"
    regex["numeric"] = "isdecimal()"
    
    regex["word"] = "\w"
    regex["non-word"] = "[^a-zA-Z0-9_ \-]"
    regex["white-spc"] = "\s"  
    regex["_"] = "_"                        
    regex["-"] = "-"
    regex["#"] = "#" 
    regex["missing"] = "isna()" 
    regex["integer"] = "isinstance(x,int)" 
    
    dateFormats = ["%m/%d/%Y","%Y-%m-%d","%Y%m%d","%m/%d/%Y %H:%M:%S %p"]
    
    nrs,ncs = df.shape

    stats,nstats,ndates,colsNoDate = dfStringAnal(df,windowP)
    
    columnSortStateTable = {}
    columnSortStateTableN = {}
    header_list = ["Column","digits","non-digits","numeric","word","non-word","white-spc","  _  ","  -  ","  #  ","% Missing","missing","string","integer","float","boolean"]
    for col in range(len(header_list)):
         columnSortStateTable[col] = True
    col_widths = [8]*len(header_list)
    col_widths[0] = 25
 
    statsVals=[]
    for col in sorted(stats.keys()):
         vals=[]
        
         vals.append(col)
         for k in header_list[1:]:
            vals.append(stats[col][k.strip()])
         statsVals.append(vals)
    slen= len(statsVals)
   
## Number Stats for Tables
    header_nlist = ["Column","Min","Max","Mean","Std","% Missing","Missing"]
    for col in range(len(header_nlist)):
        columnSortStateTableN[col] = True
        
    nstatsVals=[]
    for col in sorted(nstats.keys()):
         vals=[]
        
         vals.append(col)
         for k in header_nlist[1:]:
            vals.append(nstats[col][k])
         nstatsVals.append(vals)
    nlen=len(nstatsVals)
   
## Date Stats for Tables   
    header_dlist = ["Column","Start","End","# Unique Dates"]
    ndateVals=[]
    for col in sorted(ndates.keys()):
         vals=[]
         vals.append(col)
         for k in header_dlist[1:]:
            vals.append(ndates[col][k])
         ndateVals.append(vals)
    
   
    columns = df.columns

    colRowTable = setRowColors(statsVals,"#b3f0ff","#33d6ff","pink",header_list)
    colRowTableN = setRowColors(nstatsVals,"#8AF5A1","#2DD150","pink",header_nlist)
    
   
    layout = [ 
             [sg.Text(f"File: {file}",font="CENTAUR 15")],
             [sg.Button('Quit')],
             [sg.Multiline(default_text="Summary\n",key="-PINFO-",size=[70,5],font="CENTAUR 10")],
             [sg.Text(f"Select Identifier Column",font="CENTAUR 15"),sg.Combo(values=columns,key="-IDENT-",enable_events=False,font="CENTAUR 10"),sg.Button("IdentB")],
             [sg.Text(f"Shape : {nrs} rows  by  {ncs} columns",font="CENTAUR 15")],
             [sg.Table(values=statsVals,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,col_widths=col_widths,font="CENTAUR 10",
                   justification='center',pad=(5,5),vertical_scroll_only=False,
                   key='-TABLES-',row_colors=colRowTable,headings = header_list,metadata=[header_list,statsVals,columnSortStateTable])],
             [sg.Table(values=nstatsVals,text_color="black",
                   auto_size_columns=False,enable_events=True,num_rows=15,col_widths=col_widths,font="CENTAUR 10",justification='center',pad=(5,5),vertical_scroll_only=False,
                   key='-TABLEN-',row_colors=colRowTableN, headings = header_nlist,metadata=[header_nlist,nstatsVals,columnSortStateTableN])],
             [sg.Text("Convert Column to Date",font="CENTAUR 15"),
              sg.Combo(colsNoDate,s=(25,4),font="CENTAUR 15 bold",expand_y=True,key="-CONVDATE-",enable_events=True),
              sg.Text("Format: "),sg.Combo(dateFormats,font="CENTAUR 15 bold",key="-DATEFS-"),sg.Multiline("",s=(10,1),key="-DATEFORM-"),sg.Button("Convert to Date")],
             [sg.Table(values=ndateVals, 
                   background_color='#b3f0ff',text_color="black",
                   auto_size_columns=False,enable_events=True,num_rows=nlen+2,col_widths=[25,12,12,10],font="CENTAUR 10",justification='center',alternating_row_color='#33d6ff',pad=(5,5),vertical_scroll_only=False,
                   key='-TABLED-', headings = header_dlist,metadata=[header_dlist,ndateVals,ndates])]
             ]
    # Create the Window
    layout2 = [[sg.Column(layout,vertical_scroll_only=False,scrollable=True)]]
 
    sg.theme('Lightblue')
    window = sg.Window('Output', layout2,finalize=True,resizable=True,metadata=[df])
#    window.TKroot.focus_set()

    #window2.move(window.current_location()[0]+600, window.current_location()[1])
    table = window['-TABLES-']
    table.bind('<Button-1>', "Click")
    tablen = window['-TABLEN-']
    tablen.bind('<Button-1>', "Click")
    tabled = window['-TABLED-']
    tabled.bind('<Button-1>', "Click")

##################################################################################    
    
def dfStringShow(df,col,statType):
    if df[col].dtypes == "object":
        if statType == "digits":
          tmp = df.loc[df[col].notna() & df[col].str.contains("\d")]
        elif statType == "non-digits":
          tmp = df.loc[df[col].notna() & df[col].str.contains("\D")]
        elif statType == "numeric":
          tmp = df.loc[df[col].notna() & df[col].str.replace(".","",1).str.isdecimal()]
        elif statType == "word":
          tmp = df.loc[df[col].notna() & df[col].str.contains("\w")]
        elif statType == "non-word":
          tmp = df.loc[df[col].notna() & df[col].str.contains("[^a-zA-Z0-9_ \-]")]
        elif statType == "white-spc":
          tmp = df.loc[df[col].notna() & df[col].str.contains("\s")]
        elif statType == "_":
          tmp = df.loc[df[col].notna() & df[col].str.contains("_")]
        elif statType == "-":
          tmp = df.loc[df[col].notna() & df[col].str.contains("-")]
        elif statType == "#":
          tmp = df.loc[df[col].notna() & df[col].str.contains("#")]
        elif statType == "missing" or statType == "% miss":
          tmp = df.loc[df[col].isna()]
        elif statType == "string":
          tmp = df.loc[df[col].apply(lambda x: isinstance(x,str))]
        elif statType == "integer":
          tmp = df.loc[df[col].apply(lambda x: isinstance(x,int))]
        elif statType == "float":
          tmp = df.loc[df[col].apply(lambda x: isinstance(x,float))]
        elif statType == "boolean":
          tmp = df.loc[df[col].apply(lambda x: isinstance(x,bool))]
        
        return tmp   

#################################################################################    
    
def showDF(df,col,st,unq,regex,idcol,windowParent):
  
    h = []
    cols=[]
    fname = col.replace(" ","")  #  output name used for file if output written
    h.append("Index")
    if len(idcol) > 0:
        h.append(idcol)
        cols.append(idcol)
    h.append(col)
    cols.append(col)
   
    if len(idcol) > 0:
      tmp = df.loc[:,cols]
      values = list(zip(tmp.index.tolist(),tmp.loc[:,idcol].tolist(),tmp.loc[:,col].tolist()))
    else:
      tmp = df.loc[:,col]
      values = list(zip(tmp.index.tolist(),tmp.tolist()))
    header= h
    valuesUNQ = list(zip(unq.index.tolist(),unq.tolist()))
#    print([f"{line}\n" for line in valuesUNQ])

    titleSort=1
    hUNQ = []
    hUNQ.append("Values")
    hUNQ.append("Count")

    
    sortState = []
    for val in hUNQ:
        sortState.append(1)
    layout2 = [    
                   [sg.Text(f"Showing Rows for Column"),sg.Text(f" {col}",text_color="red"),sg.Text(f" and Reg Ex",text_color="black"),sg.Text(f"{st} : {regex[st]}",text_color="red")],
                   [sg.Text(f"Total Rows with "),sg.Text(f" {regex[st]}",text_color="red"),sg.Text(f": {tmp.shape[0]} ")],
                   [sg.Button('Quit')],
                   [sg.Button('Write Bad'),sg.Table(values=values,
                       background_color='green',vertical_scroll_only=False,font="CENTAUR 15",
                       auto_size_columns=True,enable_events=False,def_col_width=30,
                       justification='right',alternating_row_color='brown',
                       key='-TABLE2-', headings = header)],
                   [sg.Button('Write Unique'),sg.Table(values=valuesUNQ,
                       background_color='white',vertical_scroll_only=False,col_widths=60,font='Courier 10 bold ' ,
                       auto_size_columns=True,enable_events=True,def_col_width=25,
                       justification='right',alternating_row_color='tan',
                       key='-TABLE3-', headings = hUNQ)]
            ]
    window2 = sg.Window(f"DataFrame", layout2,finalize=True,resizable=True, grab_anywhere=False)
    table2 = window2['-TABLE2-']
    table3 = window2['-TABLE3-']
    table3.bind('<Button-1>', "Click")
#    window2['-TABLE2-'].expand(True, True)
    
    try:
        while True:
                event, vals = window2.read()
            #    print(event,vals)
            #    window, event, values = sg.read_all_windows()
                if event == sg.WIN_CLOSED or event == 'Quit':
                    window2.close()
            #        sys.exit(1)
                    what = "QUIT"
                    break
                elif event == "Write Unique":
                    fout = open(f"{fname}.unq.txt","a+")
                    fout.write(f"\nColumn:Type:({col},Count)\n")
                    fout.writelines([f"{col}:UNIQUE:{line}\n" for line in valuesUNQ])
                    fout.close()
                    text = f"{col} Wrote Unique Records to file {fname}.unq.txt\n"
                    windowParent["-PINFO-"].update(text,append="True")
                elif event == "Write Bad":
                    fout = open(f"{fname}.bad.txt","a+")
                    fout.write(f"Column:Type:Desc:Reg Ex:(Index,{idcol},{col})\n")
                    fout.writelines([f"{col}:BAD:{st}:{regex[st]}:{line}\n" for line in values])
                    fout.close()
                    text = f"{col} Wrote Bad Records to file {fname}.bad.txt\n"
                    windowParent["-PINFO-"].update(text,append="True")
                elif event == "-TABLE3-Click":
                    e = table3.user_bind_event
                    region = table3.Widget.identify('region', e.x, e.y)
                    if region == 'heading':
                         column = int(table3.Widget.identify_column(e.x)[1:])
                         
                         if column-1 < len(sortState):  # check to be certain column selected in range
                             sortState[column-1]*=-1
                             if sortState[column-1] == -1:
                                sortAsc=False
                             else:
                                sortAsc=True
                             valuesUNQ = sorted(valuesUNQ, key=lambda element: (element[column-1]),reverse=sortAsc)  
                             window2['-TABLE3-'].update(values=valuesUNQ)
                    elif region == 'separator':
                        continue
                    else:
                        continue
    except Exception as err:
        print(err)
        print("Table 2")

###############################################################################

def inventoryYrMoDy(df,column):
    ''' invYrMo,invYrMoDy = inventoryYrMoDy(df,"Date Column")
    Compute Year-Month and Year-Month-Day inventory counts
    for a date column in the dataframe.  It is expected the 
    column is already a PANDAS date-time object'''
    amax=df[column].max()
    amin=df[column].min()
    invYrMo = {}
    invYrMoDy = {}

    for yr in range(amin.year,amax.year+1):
        invYrMo[yr]={}
        invYrMoDy[yr]={}
        for mo in range(1,13):
            invYrMo[yr][mo]=0
            invYrMoDy[yr][mo]={}      
            for dy in range(1,32):
                invYrMoDy[yr][mo][dy]=0

    x = df[column].value_counts()
    for xx in sorted(x.index):
        d=xx
        invYrMo[d.year][d.month]+= x[xx]
        invYrMoDy[d.year][d.month][d.day]+= x[xx]   
        
    return invYrMo,invYrMoDy

###############################################################################

def showDates(df,col,windowParent):
   
    head_YrMo = ["Year","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Dec"]
    fname = col.replace(" ","")  #  output name used for file if output written
  
    iYrMo,iYrMoDy=inventoryYrMoDy(df,col)
    
    valuesYrMo=[]
    for yr in sorted(iYrMo.keys()):
        tmp = [yr]
        for mo in range(1,13):
            tmp.append(iYrMo[yr][mo])       
        valuesYrMo.append(tmp)


    
    layout2 = [    
                   [sg.Text(f"Year-Month Inventory",text_color="black"),sg.Text(f" {col}",text_color="red")],
                   [sg.Button('Quit')],
                   [sg.Button('Show YrMoDy Inv')],
                   [sg.Table(values=valuesYrMo,
                       background_color='white',vertical_scroll_only=False,
                       auto_size_columns=False,enable_events=True,def_col_width=10,font="CENTAUR 10",
                       justification='center',alternating_row_color='tan',
                       key='-TABLEDATE-', headings = head_YrMo)]
            ]
    window2 = sg.Window(f"Inventory", layout2,finalize=True,resizable=True, grab_anywhere=False,metadata=[iYrMoDy,col])
    table3n = window2['-TABLEDATE-']
    table3n.bind('<Button-1>', "Click")
#    window2['-TABLE2-'].expand(True, True)
    
############################################################################

def showYrMoDyInv(iYrMoDy,col):
    head_YrMoDy = ["Year","Month"] + [f"Day {nn}" for nn in range(1,32)]
    valuesYrMoDy=[]                             
    for yr in sorted(iYrMoDy.keys()):
        for mo in range(1,13):
            tmp = [yr,mo]
            for dy in range(1,32):
                tmp.append(iYrMoDy[yr][mo][dy])         
            valuesYrMoDy.append(tmp)
    layout3 = [    
       [sg.Text(f"Year-Month Inventory",text_color="black"),sg.Text(f" {col}",text_color="red")],
       [sg.Button('Quit')],

       [sg.Table(values=valuesYrMoDy,
           background_color='white',vertical_scroll_only=False,
           auto_size_columns=False,enable_events=True,def_col_width=8,font="CENTAUR 8",
           justification='center',alternating_row_color='tan',
           key='-TABLEDATE-', headings = head_YrMoDy)]
    ]
    window3 = sg.Window(f"Year-Month-Day Inventory", layout3,finalize=True,resizable=True, grab_anywhere=False)

def getAsciiCodes():
    codesdf = pd.read_excel("/home/joe/Python/ascii-codes.xlsx",engine='openpyxl')
    asciiCodes={}
    for index,row in codesdf.iterrows():
        asciiCodes[row['DEC']] = {}
        asciiCodes[row['DEC']]['symbol'] = row['Symbol']
        asciiCodes[row['DEC']]['desc'] = row['Description']
    return asciiCodes

def analyzeString(line,nchars,histo,histc):
    for c in line:
        n = ord(c)
        nchars+=1
        if n not in histo:
            histo[n]=0
        histo[n]+=1    
        if c not in histc:
            histc[c]=0
        histc[c]+=1    
    return nchars
   
def fileAnalyze(file,asciiCodes):
    nlines=0
    nchars=0
    histo={}
    histc={}
    nlines=0
    
    try: 
        fin = open(file,"r")
        for line in fin:
            nlines+=1
            nchars=analyzeString(line,nchars,histo,histc)

    except:
        print("Using Latin Encoding")
        fin = open(file,"r",encoding='latin')
        for line in fin:
            nlines+=1
            nchars=analyzeString(line,nchars,histo,histc)
     

    print(f"Total Characters: {nchars:,d}")
    print(f"Total Lines     : {nlines:,d}")
    print("Char   Count")
    print("----   -----")
    data = []
    for n,count in sorted(histo.items()):
        print(f"{n:4d}  {count:6d}  {asciiCodes[n]['symbol']} - {asciiCodes[n]['desc']}")
        tmp=[]
        tmp.append(n)
        tmp.append(count)
        tmp.append(asciiCodes[n]['symbol'])
        tmp.append(asciiCodes[n]['desc'])
        data.append(tmp)
    return nchars,nlines,data
    
    
def analizeFileCharacters(event,values,color1,color2):
   asciiCodes = getAsciiCodes()
   if event == "-FILE1-" and len(values["-FILE1-"]) > 0:
         file = values["-FILE1-"]
         nchars,nlines,data = fileAnalyze(file,asciiCodes)
   else:
       return
   
   header_list = ["Character Code","Count","Symbol","Definition"]
   col_widths = [8]*len(header_list)
   col_widths[-1] = 30
   colrs =[color1,color2]
   colTab = [colrs[nn%2] for nn in range(len(data))]
   rowNums = [num for num in range(0,len(data)+1)]
   colText = ["black"]*len(colTab)
   colrw = list(zip(rowNums,colTab))
 
    
   layout = [[sg.Text(f"Character Analysis for File: {file}",font="CENTAUR 15")],
             [sg.Text(f"Total Characters: {nchars:,d}",font="CENTAUR 15")],
             [sg.Text(f"Total Lines: {nlines:,d}",font="CENTAUR 15")],
             [sg.Button("Quit")],
             [sg.Table(values=data,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,font="CENTAUR 15",
               justification='right',pad=(5,5),vertical_scroll_only=False,row_colors=colrw,
               key='-CHARS-',col_widths=col_widths,headings = header_list)
              ]            
            ]
      
   
   layout2 = [[sg.Column(layout,vertical_scroll_only=False)]]
    
                  
              
   window2 = sg.Window('Character Analysis',layout2,finalize=True,resizable=True)

#################################################################################

def analizeColsPerRow(event,values,color1,color2):

   if event == "-FILE1-" and len(values["-FILE1-"]) > 0:
         file = values["-FILE1-"]
   else:
       return
    
   if file[-4:] == ".tsv":
    delim="\t"
   elif file[-4:] == ".csv":
        delim=","
   elif file[-5:] == ".xlsx" or file[-4:] == ".xls":
        print("Cannot Perform Column vs Row Analysis on Excel Files.... Sooooo Sorry...NOT")
        print("Cannot Perform Column vs Row Analysis on Excel Files.... Sooooo Sorry...NOT")
        print("Cannot Perform Column vs Row Analysis on Excel Files.... Sooooo Sorry...NOT")
        return    
   else:
        print("Unknow File Extension, cannot perform Column vs Row Analysis...How did that slip in?")
        print("Unknow File Extension, cannot perform Column vs Row Analysis...How did that slip in?")
        print("Unknow File Extension, cannot perform Column vs Row Analysis...How did that slip in?")
        return
    
   try:
      fin=open(file,encoding="latin")
   except Exception as err:
      print("Unable to open Text file... Trying LATIN encoding")
      fin=open(file,encoding="latin")
   nlines=0
   hist={}
   for line in fin:
        nlines+=1
        spl=line.split(delim)
        if nlines > 1:
            nn=len(spl)
            if nn in hist:
                hist[nn]+=1
            else:
                hist[nn]=1
        else:
            headerLength=len(spl)
            headerCols=spl
   fin.close()
   headerTable = ["# of Columns","# of Occurrences"]
   data = []
   for ncols,noc in sorted(hist.items()):
       tmp=[ncols,noc]
       data.append(tmp)
              
   print("COL DATA ",data)
   col_widths = [15]*len(headerTable)
   col_widths[-1] = 30
   colrs =[color1,color2]
   colTab = [colrs[nn%2] for nn in range(len(data))]
   rowNums = [num for num in range(0,len(data)+1)]
   colText = ["black"]*len(colTab)
   colrw = list(zip(rowNums,colTab))
 
    
   layout = [[sg.Text(f"Columns Per Row Analysis for File: {file}",font="CENTAUR 15")],
             [sg.Text(f"Total Lines (including header): {nlines:,d}",font="CENTAUR 15")],
             [sg.Text(f"# of Columns in Header: {len(headerCols)}",font="CENTAUR 15")],
                       
             [sg.Button("Quit")],
             [sg.Table(values=data,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,font="CENTAUR 15",
               justification='right',pad=(5,5),vertical_scroll_only=False,row_colors=colrw,
               key='-COLSPERROW-',col_widths=col_widths,headings = headerTable)
              ]            
            ]
      
   
   layout2 = [[sg.Column(layout,vertical_scroll_only=False)]]
    
                  
              
   window2 = sg.Window('Character Analysis',layout2,finalize=True,resizable=True)

####################################################################################

def showRecords(df,file,first=10,last=0):
   
   header_list = ["Index"] +  list(df.columns)
   col_widths = [8]  #  set the first column width for the index
   for col in df.columns:
      if df[col].dtype == "O":
         nn = df.loc[:10,col].str.len().max()
         print("WIDTHS ",col,nn)
      else:
         nn=8
      print("NN ",int(nn))
      col_widths.append(int(nn))
   print("WIDTHS ",col_widths)
   data = df.values[:10].tolist()
   indxs = df1.index.values[0:10].tolist()
   for indx,rows in enumerate(data):
      rows.insert(0,indxs[indx])
      
#   col_widths = [8]*len(header_list)
 #  col_widths[-1] = 30
   colrs =[color1,color2]
   colTab = [colrs[nn%2] for nn in range(len(data))]
   rowNums = [num for num in range(0,len(data)+1)]
   colText = ["black"]*len(colTab)
   colrw = list(zip(rowNums,colTab))
    
   layout = [[sg.Text(f"First 10 records for File: {file}",font="CENTAUR 15")],
             [sg.Text(f"Total Columns: {df.shape[1]:,d}",font="CENTAUR 15")],
             [sg.Text(f"Total Lines: {df.shape[0]:,d}",font="CENTAUR 15")],
             [sg.Checkbox("Show First Records",font="CENTAUR 15",key="-SHOWFIRSTRECS-"),sg.Text("# of lines to show:",font="CENTAUR 15",),sg.Input("",size=3,font="CENTAUR 15",key="-NFIRSTRECS-")],
             [sg.Checkbox("Show Last Records",font="CENTAUR 15",key="-SHOWLASTRECS-"),sg.Text("# of lines to show:",font="CENTAUR 15",),sg.Input("",size=3,font="CENTAUR 15",key="-NLASTRECS-")],
             [sg.Button("Update",key="-UPDATESHOWRECS-")],
             [sg.Button("Quit")],
             
             [sg.Table(values=data,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,font="CENTAUR 15",
               justification='right',pad=(5,5),vertical_scroll_only=False,row_colors=colrw,
               key='-RECORDS-',col_widths=col_widths,headings = header_list)
              ] 
            ]
      
   
   layout2 = [[sg.Column(layout,vertical_scroll_only=False)]]
    
                  
              
   window2 = sg.Window('Records in File',layout2,finalize=True,resizable=True,metadata=df)

##########################################################################

def decodeSIJ(file):
    fin = open(file,"r")
    
    infoSIJ = {}
    data = json.load(fin)
   
    fin.close()
    data['datasetID'],data['fileToPublish'],data['pathToSavedFile']
    infoSIJ['4x4']=data['datasetID']
    infoSIJ['file']=data['fileToPublish']
    infoSIJ['path']=data['pathToSavedFile']
    
    iFile = json.loads(data['controlFileContent'])
    for key,val in iFile.items():
        if 'columns' in val:
            inCols = val['columns']
    infoSIJ['inputColumns'] = inCols
    
    oFile=json.loads(data['ftpControlFileContent'])
    for key,val in oFile.items():
    #    display(key,val)
        if 'columns' in val:
            outCols = val['columns']
    infoSIJ['outputColumns'] = inCols
    return infoSIJ

###########################################################################################

def showSIJ(file,info):
   
   header_list = ["Index","Input Columns","CIM Columns"]
   mlen = len(max([ max(info['inputColumns'],key=len),max(info['outputColumns'],key=len)]))
   col_widths = [8,mlen,mlen]  #  set the first column width for the index
   indxs = [nn+1 for nn in range(len(info['inputColumns']))]
   data=list(zip(indxs,info['inputColumns'],info['outputColumns']))
 
#   col_widths = [8]*len(header_list)
 #  col_widths[-1] = 30
   colrs =[color1,color2]
   colTab = [colrs[nn%2] for nn in range(len(data))]
   rowNums = [num for num in range(0,len(data)+1)]
   colText = ["black"]*len(colTab)
   colrw = list(zip(rowNums,colTab))
    
   layout = [[sg.Text(f"SIJ File: {file}: ",font="CENTAUR 15")],
             [sg.Text(f"4x4: {info['4x4']}: ",font="CENTAUR 15")],
             [sg.Text(f"DATA File: {info['file']}",font="CENTAUR 15")],
             [sg.Text(f"DATA Path: {info['path']}",font="CENTAUR 15")],
             
             [sg.Button("Quit")],
             
             [sg.Table(values=data,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,font="CENTAUR 15",
               justification='right',pad=(5,5),vertical_scroll_only=False,row_colors=colrw,
               key='-SIJColumns-',col_widths=col_widths,headings = header_list)
              ] 
            ]
      
   
   layout2 = [[sg.Column(layout,vertical_scroll_only=False)]]
    
                  
              
   window2 = sg.Window('SIJ Info',layout2,finalize=True,resizable=True,metadata=info)

## GUI

In [ ]:

#############################################
def compareWindow(stats1,df1,file1):
    global color1,color2
    header_list = ["Column","% Missing","Missing","string","integer","float","boolean"]
    col_widths = [8]*len(header_list)
    col_widths[0] = 25
    columnSortStateTable1 = {}
  
    print("Comparing Data")
    ## set up sort state for the column in both tables
    for col in header_list:
         columnSortStateTable1[col] = -1
       
            
    valsFile1 = getValues(stats1,header_list)
   
    print("LAYOUT")
    rowFile1Colors = setRowColors(valsFile1,color1,color2,"pink",header_list)

    
    layCol1 = [[sg.Text(f"File 1 {file1}",font="CENTAUR 15")],[sg.Text(f"File 1 Shape {df1.shape}",font="CENTAUR 15")],
               [sg.Button("Detailed Analysis",font="CENTAUR 15")],
               [sg.Table(values=valsFile1,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,col_widths=col_widths,font="CENTAUR 10",
                   justification='center',pad=(5,5),vertical_scroll_only=False,
                   key='-TABLEFILE-',row_colors=rowFile1Colors,headings = header_list,metadata=columnSortStateTable1)]
               ]
    
    

    layout = [[sg.Button("Quit")],
              
              [sg.Text("Change Header Row",font="CENTAUR 10"),
              sg.Combo([1,2,3,4,5,6,7,8,9,10],font="CENTAUR 10",enable_events=True,key="-HEADERCOMBO-"),sg.Text(" (Header Count is 0 based) ",font="CENTAUR 10"),
              sg.Button("Change Header")],
              [sg.Text(f"File 1 {file1}",font="CENTAUR 15")],[sg.Text(f"File 1 Shape {df1.shape}",font="CENTAUR 15")],
               [sg.Button("Detailed Analysis",font="CENTAUR 15")],
               [sg.Table(values=valsFile1,text_color="black", auto_size_columns=False,enable_events=True,num_rows=20,col_widths=col_widths,font="CENTAUR 10",
                   justification='center',pad=(5,5),vertical_scroll_only=False,
                   key='-TABLEFILE-',row_colors=rowFile1Colors,headings = header_list,metadata=columnSortStateTable1)
               ]            
  #            [layCol1]  
             ]
    
   
    layout2 = [[sg.Column(layout,vertical_scroll_only=False)]]
    
                  
              
    window2 = sg.Window('Compare',layout2,finalize=True,resizable=True,metadata=[df1,stats1,file1])
    # a = window2.CurrentLocation()
    # screen_width, screen_height = window2.get_screen_dimensions()
    # win_width, win_height = window2.size
    # x, y = (screen_width - win_width)//2, (screen_height - win_height)//2
    # x=200
    # y=200
    # window2.move(x, y)
    tableFile1 = window2['-TABLEFILE-']
    tableFile1.bind('<Button-1>', "Click")
   
    return window2,tableFile1,valsFile1,rowFile1Colors

##############################################   

def fileBrowser():
    global df,df2,numWindows,color1,color2,df1
    global stats1,stats2,window,statsVals
    dataStore = {}
    headerStore = {}
    header_list = ["Column","% Missing","Missing","string","integer","float","boolean"]
    
    regex = {}
    regex["digits"] = "\d"
    regex["non-digits"] = "\D"
    regex["numeric"] = "isdecimal()"
    
    regex["word"] = "\w"
    regex["non-word"] = "[^a-zA-Z0-9_ \-]"
    regex["white-spc"] = "\s"  
    regex["_"] = "_"                        
    regex["-"] = "-"
    regex["#"] = "#" 
    regex["missing"] = "isna()" 
    regex["integer"] = "isinstance(x,int)" 
    
    
    
    
    col_widths = [8]*len(header_list)
    col_widths[0] = 25
    columnSortStateTable1 = {}
    columnSortStateTable2 = {}

    ## set up sort state for the column in both tables
    for col in header_list:
         columnSortStateTable1[col] = -1
         columnSortStateTable2[col] = -1    
    
    # previousFiles = getPrevFiles(1)
    # previousFiles = list(previousFiles.keys())
    # print("Create Layout")
    layout = [[sg.Button("Close")],
          ##   [sg.Button("Compare")],
             [sg.FilesBrowse(button_text="Read File",initial_folder="/home/joe/bic_etl",font="CENTAUR 15",file_types=[("TSV Files","*.tsv"),("CSV Files","*.csv"),("Excel Files","*.xlsx"),("Excel Files","*.xls"),("SIJ Files","*.sij")],enable_events=True,key='-FILE1-')],
             [ sg.Checkbox("No Quoting",font="CENTAUR 15",key="-QUOTENO-")],
             [sg.Text("Web Address:",font="CENTAUR 15"),sg.Input(default_text="",font="CENTAUR 15",key="-WEB1-"),sg.Button("Fetch")], 
             [sg.Checkbox("Show Records",font="CENTAUR 15",key="-SHOWRECS-")],
             [sg.Checkbox("Character Analysis",font="CENTAUR 15",key="-CHARANAL-")],
             [sg.Checkbox("Cols per Row",font="CENTAUR 15",key="-COLSROW-")],
             [sg.Checkbox("Error Analysis",font="CENTAUR 15",key="-ERRORANAL-")],
             [sg.Multiline(default_text="Summary\n",key="-PINFO-",font="CENTAUR 15",size=[70,5])]
   #          [sg.Output(size=(40,10),font="CENTAUR 15")],
             ]
              
    layout2 = [[sg.Column(layout,vertical_scroll_only=False,scrollable=True,size=(715,407))]]
    window = sg.Window('Files',layout2,finalize=True,resizable=True,size=(715,607))
    
    a = window.CurrentLocation()
    print("STARTING")
    
    # screen_width, screen_height = window.get_screen_dimensions()
    # win_width, win_height = window.size
    # x, y = (screen_width - win_width)//2, (screen_height - win_height)//2
 
    x=200
    y=200
 #   window.move(x, y)
    windowsOpen = {}
    windowsOpen["main"] = []
    windowsOpen["unique"] = []
    numWindows=0
    print("Going to Loop")
    while True:
        try:
            wid, event, values = sg.read_all_windows()
            
            print("MAIN ",wid)
            print("MAIN ",event)
            print("MAIN ",values)
            if event == sg.WIN_CLOSED or event == 'Close':
            
                break
            elif event == event == 'Quit':
                wid.close()
                
              #  break
            elif event == "-FILE1-" or event == "Compare" or event == "Fetch":
     #           files=[]
#                 if event == "Compare":
# #                    values["-FILE1-"] = "/home/joe/bic_etl/cdos/business/nonprofit/data_transformed/sol_typ_entity.csv"
#                     values["-FILE1-"] = "/home/joe/bic_etl/cdor/regulations_liquor/data_transformed/liquorLicenses.csv"
                quoting=0
                if values["-QUOTENO-"]:
                    quoting=3
                    print("QUOTING QUOTING",quoting)
                if values["-ERRORANAL-"]:
                    file=values["-FILE1-"]
            #        file="/home/joe/work/COURT/CountyCourtFilings.csv"
                    nlines,delim,columns = fileAnalysis(file)
                   
                    string = f"file: {file}\n# of lines: {nlines}\nDelimiter: {delim}\n"
                    for k,v in columns.items():
                        string+= f" {k} # Columns, # of lines {v}\n"
                    string+= "\n-----------------------------------------------\n"
                    window["-PINFO-"].update(string)
                else:
                    color1 = colorPairs[numWindows][0]
                    color2 = colorPairs[numWindows][1]
                    print("QUOTI MAIN ",quoting)
                    numWindows+=1
                    if numWindows > len(colorPairs):
                        numWindows=0
                    if (event == "-FILE1-" and values["-FILE1-"][-4:].lower() != ".sij") or event == "Fetch":
                        print("VALUES ",values)
                        df1,stats1,file1 = getFilesClicked(event,values,window,quoting)
                      

                        WindowC,tabl1,valsFile1,rowFile1Colors = compareWindow(stats1,df1,file1)
                        dataStore[tabl1]=valsFile1
                        windowsOpen["main"].append(WindowC)

                        if values["-CHARANAL-"]:
                            print("ANALZE Chars")
                            analizeFileCharacters(event,values,color1,color2)
                        if values["-COLSROW-"]:
                            print("ANALZE Columns per Row")
                            analizeColsPerRow(event,values,color1,color2)

                        if values["-SHOWRECS-"]:
                            print("SHOW RECS")
                            showRecords(df1,file1)
                    else:  #  SIJ file
                        file = values["-FILE1-"]
                        infoSIJ = decodeSIJ(file)
                        showSIJ(file,infoSIJ)

###  Update and add more records to show                        
            elif event == "-UPDATESHOWRECS-": 
                df = wid.metadata
                print("UPDATE RECRODS DF",df.head())
                first=0
                data=[]
                if values["-SHOWFIRSTRECS-"]:
                    first=int(values["-NFIRSTRECS-"])
                    data = df.values[:first].tolist()
                    indxs = df.index.values[0:first].tolist()
                    for indx,rows in enumerate(data):
                        rows.insert(0,indxs[indx])
                if values["-SHOWLASTRECS-"]:
                    last=int(values["-NLASTRECS-"])
                    if first > 0:
                         tmp = df.values[-last:].tolist()
                         indxs = df.index.values[-last:].tolist()
                         for indx,rows in enumerate(tmp):
                            rows.insert(0,indxs[indx])
                         data = data+tmp
                    else:
                         data = df.values[-last:].tolist()
                         indxs = df.index.values[-last:].tolist()
                         for indx,rows in enumerate(data):
                            rows.insert(0,indxs[indx])

                if len(data) > 0:
                   colrs =[color1,color2]
                   colTab = [colrs[nn%2] for nn in range(len(data))]
                   rowNums = [num for num in range(0,len(data)+1)]
                   colrw = list(zip(rowNums,colTab))
                   wid['-RECORDS-'].update(values=data,row_colors=colrw)
                         
                    
   #             showRecords(df1,file1,first,last,update)   
                
### Write Unique Values
            elif event == "Write Unique":
                fdir = sg.popup_get_folder("Choose Folder",initial_folder="/home/joe",font="CENTAUR 15")
                outFile = sg.popup_get_text("Enter Output Filename",font="CENTAUR 15")
                fo=f"{fdir}/{outFile}"
                print("FO FO FO FO ",fo)
                vals=wid.metadata[0]
                print("VALS ",vals)
                origCol=wid.metadata[1]
                origFile=wid.metadata[2]
                print(f"Source File: {origFile}\n")
                print(f"Column: {origCol}\n\n")
                try: 
                    fout = open(fo,"w")
                    fout.write(f"Source File: {origFile}\n")
                    fout.write(f"Column: {origCol}\n\n")

                    fout.write(f"\nColumn:Type:({col},Count)\n")
                    fout.writelines([f"{line}\n" for line in vals])
                    fout.close()
                except Exception as err:
                    print("ERROR ",err)
                text = f"{col} Wrote Unique Records to file {fdir}/{outFile}\n"
                window["-PINFO-"].update(text,append="True")

### Table Click
            elif event == "-TABLEFILE-Click":
               
                row,col=getRowClicked(tabl1,valsFile1)
                stats1=wid.metadata[1]
                if row == 0:
                   valsFile1,columnSortStateTable1 = sortTable(row,stats1,columnSortStateTable1,tabl1,event,header_list)          
                   rowFile1Colors= setRowColors(valsFile1,color1,color2,"pink",header_list)
                  
                   wid["-TABLEFILE-"].update(values=valsFile1,row_colors=rowFile1Colors)
                else:
                    unq = df1[col].value_counts()
                    w,t,values,uHead = showDFUN(col,unq,wid,"#b3f0ff","#33d6ff")
                    dataStore[t] = values
                    headerStore[t] = uHead
                    windowsOpen["unique"].append(w)
                   
            elif event == "-TABLE-Click":
                table = wid['-TABLE-']
                
                sortUniqe(table,wid,dataStore,headerStore)

### Change HEADER   
            elif event == "Change Header":
                headerRow = int(values["-HEADERCOMBO-"])
                file1 = wid.metadata[2]              
                df1 = getFile("Local",file1,window,quoting,headerRow)              
                stats1 = dfAnalyze(df1)
                wid.close()
                WindowC,tabl1,valsFile1,rowFile1Colors = compareWindow(stats1,df1,file1)
                dataStore[tabl1]=valsFile1
                windowsOpen["main"].append(WindowC)
                                
### Detailed Analysis
            elif event == "Detailed Analysis":    
                dfO = wid.metadata[0]
                file = wid.metadata[2]
                
                
                showDetailedAnalysis(dfO,window,file,idcol="")

### Detailed Clicked            
            elif event in ['-TABLES-Click','-TABLEN-Click']:
                ts = event.replace("Click","")
                
                table = wid[ts]
                header = table.metadata[0]
                statsVals = table.metadata[1]
                df = wid.metadata[0]
                e = table.user_bind_event
                region = table.Widget.identify('region', e.x, e.y)
                if region == 'heading':
                    row = 0
                elif region == 'cell':
                    row = int(table.Widget.identify_row(e.y))
                elif region == 'separator':
                    continue
                else:
                    continue
                    
                if row > 0:
                    colClicked = int(table.Widget.identify_column(e.x)[1:])
                    statClicked = header[colClicked-1].strip()
                    columnClicked = statsVals[row-1][0] 
                   
                    tmp = dfStringShow(df,columnClicked,statClicked)             
                    un = df[columnClicked].value_counts()
                    idcol=""
                    showDF(tmp,columnClicked,statClicked,un,regex,idcol,window)
                else:
                    colClicked = int(table.Widget.identify_column(e.x)[1:])-1
                   
                    statClicked = header[colClicked].strip()
                   
                    columnSortState = table.metadata[2]
                    columnSortState[colClicked]= not columnSortState[colClicked]
                    statsS =     sorted(statsVals, key=lambda x: x[colClicked],reverse=columnSortState[colClicked])
                    colorsTable = setRowColors(statsS,"#b3f0ff","#33d6ff","pink",header)
                    table.metadata = [header,statsS,columnSortState]
                    wid[ts].update(values=statsS,row_colors=colorsTable)     
                
### Convert Date
            elif event == 'Convert to Date':      
                print("Converting Date") 
                dcol = values["-CONVDATE-"]
                dform = values["-DATEFORM-"]
                if len(dform) > 0:
                    form=dform.strip()
                else:
                    form= values["-DATEFS-"]
                    
                df = wid.metadata[0]
                df[dcol]=pd.to_datetime(df[dcol], format=form)
                if form.find("H") > -1 or form.find("S") > -1:
                    df[dcol] = df[dcol].dt.normalize()
                ndates = wid["-TABLED-"].metadata[2]
                header = wid["-TABLED-"].metadata[0]
                
                ndates[dcol]={}
                ndates[dcol]["Start"] = df[dcol].min()
                ndates[dcol]["End"] = df[dcol].max()
                ndates[dcol]["# Unique Dates"] = df[dcol].nunique()
                ndateVals=[]
                for col in sorted(ndates.keys()):
                     vals=[]
                     vals.append(col)
                     for k in header[1:]:
                        vals.append(ndates[col][k])
                     ndateVals.append(vals)
                wid['-TABLED-'].update(values=ndateVals)
                wid['-TABLED-'].metadata = [header,ndateVals,ndates]
                
### TableD Clicked
            elif event == '-TABLED-Click':
                table = wid["-TABLED-"]
                e = table.user_bind_event
                region = table.Widget.identify('region', e.x, e.y)
                if region == 'heading':
                    pass
                elif region == 'cell':
                    row = int(table.Widget.identify_row(e.y))
                elif region == 'separator':
                    continue
                else:
                    continue
                ndateVals = wid["-TABLED-"].metadata[1]
                colClicked = ndateVals[row-1][0]
                df=wid.metadata[0]
                
                # tmp = dfStringShow(septicOrig,columnClicked,statClicked)
                showDates(df,colClicked,window)

### Show YrMoDy Inventory
            elif event == 'Show YrMoDy Inv':
                iYrMoDy = wid.metadata[0]
                col = wid.metadata[1]
                showYrMoDyInv(iYrMoDy,col)
            
        
        except Exception as err:
             exceptionLog(err,inspect.currentframe().f_code.co_name)
    for wid in windowsOpen["unique"]:
        
        if wid:
            print("cosing ",wid)
            print(wid.close())
            wid = None
            

    for wid in windowsOpen["main"]:
       
        if wid:
            print("cosing ",wid)
            wid.close()
            wid = None
          
    window.close()

print("Lets Go")
fileBrowser()

Lets Go
STARTING
Going to Loop
MAIN  <PySimpleGUI.PySimpleGUI.Window object at 0x7fc607899290>
MAIN  Fetch
MAIN  {'-FILE1-': '', '-QUOTENO-': False, '-WEB1-': 'https://data.colorado.gov/resource/4ykn-tg5h.csv', '-SHOWRECS-': False, '-CHARANAL-': False, '-COLSROW-': False, '-ERRORANAL-': False, '-PINFO-': 'Summary'}
QUOTI MAIN  0
VALUES  {'-FILE1-': '', '-QUOTENO-': False, '-WEB1-': 'https://data.colorado.gov/resource/4ykn-tg5h.csv', '-SHOWRECS-': False, '-CHARANAL-': False, '-COLSROW-': False, '-ERRORANAL-': False, '-PINFO-': 'Summary'}
Get Files
QUOTING  0
Fetch
<class 'dict'> <class 'str'>
48
{'-FILE1-': '', '-QUOTENO-': False, '-WEB1-': 'https://data.colorado.gov/resource/4ykn-tg5h.csv', '-SHOWRECS-': False, '-CHARANAL-': False, '-COLSROW-': False, '-ERRORANAL-': False, '-PINFO-': 'Summary'}
Getitng FIle  https://data.colorado.gov/resource/4ykn-tg5h.csv
GETTING FETCH FILE


## EXTRA

In [7]:
df=pd.read_csv("https://data.colorado.gov/resource/885z-76pr.csv")

HTTPError: HTTP Error 403: Forbidden

In [8]:
df.shape

(614238, 24)

In [9]:
df.columns

Index(['facilityid', 'facilityname', 'streetnumber', 'streetdirection',
       'streetname', 'streettype', 'streetunit', 'city', 'state', 'zip',
       'typeoffacility', 'categoryoffacility', 'inspectiontype',
       'inspectiondate', 'violationcode', 'violation', 'violationpoints',
       'violationtype', 'violationstatus', 'inspectionscore', 'location',
       'siteaddress', 'facilitycategory', 'georeference'],
      dtype='object')

In [10]:
df['categoryoffacility'].value_counts()

categoryoffacility
RESTAURANT 0 TO 100 SEATS                     328188
NO FEE LICENSE K12 SCHOOLS NON PROFIT          71750
RESTAURANT 101 TO 200 SEATS                    58840
LIMITED FOOD SERVICE CONVENIENCE OTHER         47561
MOBILE UNIT FULL FOOD SERVICE                  21123
RESTAURANT MORE THAN 200 SEATS                 19715
GROCERY STORE W DELI 0 TO 15000 SQ FT          15075
GROCERY STORE W DELI MORE THAN 15000 SQ FT     13238
SPECIAL EVENT                                  13061
GROCERY STORE 0 TO 15000 SQ FT                 12177
MOBILE UNIT PREPACKAGED                         5912
TEMP EVENT ON SITE PREP                         5436
GROCERY STORE  MORE THAN 15000 SQ FT            2162
Name: count, dtype: int64

In [11]:
df['facilitycategory'].value_counts()

facilitycategory
FAST FOOD LIMITED MENU          168970
FULL SERVICE FULL MENU          160327
FULL MENU LIMITED SERVICE        77530
SCHOOLS                          63577
GROCERY FINISHED FOODS           38443
CONVENIENCE STORES               24933
MOBILE UNITS                     21478
SPECIAL EVENT                    13173
CONCESSIONS SENIOR NUTRITION     11480
CATERING                          6050
PRE PACKAGED                      5966
BARS FRATERNAL ORGANIZATIONS      5564
TEMPORARY EVENTS                  5436
RESIDENTIAL FACILITIES            4422
CHURCHES                          2394
RETAIL COMMISSARY                 2222
FOOD BANK                         1975
ONLINE DELIVERY                    298
Name: count, dtype: int64

In [3]:
new=pd.read_csv("/home/joe/bic_etl/boulder/data_source/restaurant_inspections.csv")

/tmp/ipykernel_11721/2361306772.py:1: DtypeWarning: Columns (3,10) have mixed types. Specify dtype option on import or set low_memory=False.
  new=pd.read_csv("/home/joe/bic_etl/boulder/data_source/restaurant_inspections.csv")


In [6]:
df.shape

(1000, 24)

In [4]:
new.columns

Index(['FACILITY_ID', 'FACILITY_NAME', 'SITE_ADDRESS', 'STREET_NUMBER',
       'STREET_DIRECTION', 'STREET_NAME', 'STREET_TYPE', 'STREET_UNIT', 'CITY',
       'STATE', 'ZIP', 'TYPE_OF_FACILITY', 'CATEGORY_OF_FACILITY',
       'INSPECTION_TYPE', 'INSPECTION_DATE', 'VIOLATION_CODE', 'VIOLATION',
       'VIOLATION_POINTS', 'VIOLATION_TYPE', 'VIOLATION_STATUS',
       'INSPECITON_SCORE'],
      dtype='object')

In [8]:
file="/home/joe/work/Restaurant_Inspections_in_Boulder_County_20250220.csv"
df=pd.read_csv(file)

/tmp/ipykernel_769/2285926368.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file)


In [9]:
df.columns

Index(['facilityId', 'facilityName', 'siteAddress', 'streetNumber',
       'streetDirection', 'streetName', 'streetType', 'streetUnit', 'city',
       'state', 'zip', 'typeOfFacility', 'categoryOfFacility',
       'inspectionType', 'inspectionDate', 'violationCode', 'violation',
       'violationPoints', 'violationType', 'violationStatus',
       'inspectionScore', 'location'],
      dtype='object')

In [20]:
df.isna().sum()

facilityId                 0
facilityName               0
siteAddress                0
streetNumber             413
streetDirection       470989
streetName                 0
streetType             12618
streetUnit            430581
city                       0
state                      0
zip                        0
typeOfFacility             0
categoryOfFacility         0
inspectionType             0
inspectionDate             0
violationCode              0
violation                  0
violationPoints            1
violationType         120149
violationStatus        56695
inspectionScore            0
location                   0
dtype: int64

In [19]:
print(df.loc[df["city"].str.lower() == "lincoln",["facilityName","siteAddress","streetNumber","streetName","zip"]])
print(df["city"].value_counts())

                    facilityName       siteAddress streetNumber streetName  \
172090  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN       1821.0     URBANA   
172094  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN       1821.0     URBANA   
172103  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN       1821.0     URBANA   
172111  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN       1821.0     URBANA   
172121  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN       1821.0     URBANA   
...                          ...               ...          ...        ...   
408056  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN         1821     URBANA   
408067  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN         1821     URBANA   
408073  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN         1821     URBANA   
408074  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN         1821     URBANA   
408075  DIPPIN DOTS (TEMP EVENT)  1821 N URBANA LN         1821     URBANA   

          zip  
172090  68505  
172094  68505  
172103  68505  

In [ ]:
dfO = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv",delimiter="\t",encoding="latin")
dfT = pd.read_csv("/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.csv")



In [27]:
file="/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv"
quoting=0
header=0
delim="\t"
pd.read_csv(file,delimiter="\t")

,farmProduct,debtorName,debtorId,counties,cropYear,debtorAddress,additionalDebtors,additionalDebtorId,securedParty,assignee,additionalFarmProduct,recordIdVerbatim,recordId,recordIdDate,recordCountyFilingId,recordCountyFiling,amendmentIdVerbatim,amendmentId
All Field Crops,100X FARM,6.066242e+09,Boulder; Larimer; Weld,ALL,"1716 MAIN ST, LONGMONT, CO, 80501",BLOOMSTEADS; ID #: 6126022194. CULINARY FARMST...,6126022194;6216032194;6016132034;2016136039;80...,FARM SERVICE AGENCY for UNITED STATES OF AMERI...,NONE,NaN,20222046557 - 05/05/2022,20222046557,05/05/2022,NaN,NaN,20232038290 - 04/20/2023,20232038290,NaN
All Field Crops,1708 FARMS GP,7.012061e+09,Yuma,ALL,"38348 County Road D40, Yuma, CO, 80759","MONK, TRENT LANE; ID #: 1153135200. NEILLMONK,...",1153135200;0050141100,"TBK Bank SSB, 100 West Pearl, Lamar, CO, 81052",NONE,All Livestock,20232045569 - 05/10/2023,20232045569,05/10/2023,NaN,NaN,NaN,NaN,NaN
All Field Crops,17 SQUARED LLC,5.179196e+09,Bent,ALL,"15030 US Highway 50, Las Animas, CO, 81054",NaN,NaN,"McClave State Bank, PO Box 3, 101 E 1st Street...",NONE,Corn; Cotton (Field Or Row Crop); Hay; Oats; O...,20242058636 - 06/26/2024,20242058636,06/26/2024,NaN,NaN,NaN,NaN,NaN
All Field Crops,376 FISH INC,9.406804e+07,Adams; Arapahoe; Boulder; Broomfield; Denver; ...,ALL,"3812 Howe Ct, Boulder, CO, 80301","KILT FARM; ID #: 0094118137. MOSS, MICHAEL A; ...",0094118137;2158137134,Farm Service Agency acting for the United Stat...,NONE,All Vegetables; Honey,20162023906 - 03/16/2016,20162023906,03/16/2016,NaN,NaN,20162024080 - 03/17/2016; 20212013535 - 02/11/...,20162024080;20212013535,NaN
All Field Crops,3D FARMS LLC,NaN,Prowers,ALL,"36683 Highway 63, Akron, CO, 80720","DRACON, JOEL WILLIAM; ID #: 1187046103. DRACON...",1187046103;3183046117;4185049136,"TBK Bank, SSB, 100 West Pearl, Lamar, CO, 81052",NONE,All Livestock,20242035686 - 04/22/2024,20242035686,04/22/2024,NaN,NaN,20242065504 - 07/18/2024,20242065504,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wool,THEOS SWALLOW FORK RANCH LLP,8.020016e+07,Rio Blanco,ALL,"NO ADDRESS, NO CITY, NO STATE, NO ZIP",THEOS SWALLOW FORK RANCH LLP; ID #: 1088201167...,1088201167;2081200167;9083205010;2081200207,"BANK OF THE SAN JUANS, DIVISION OF GLACIER BAN...",NONE,Sheep And Lamb,2003F115913 - 10/23/2003,2003F115913,10/23/2003,NaN,NaN,2003F115959 - 10/23/2003; 2008F077472 - 08/01/...,2003F115959;2008F077472;20182091976;2013206645...,NaN
Wool,THEOS SWALLOW FORK RANCH LLP,1.088201e+09,Rio Blanco,ALL,"PO BOX 195, MEEKER, CO, 81641",THEOS SWALLOW FORK RANCH LLP; ID #: 0080200160...,0080200160;2081200167;9083205010;2081200207,"BANK OF THE SAN JUANS, DIVISION OF GLACIER BAN...",NONE,Sheep And Lamb,2003F115913 - 10/23/2003,2003F115913,10/23/2003,NaN,NaN,2003F115959 - 10/23/2003; 2008F077472 - 08/01/...,2003F115959;2008F077472;20182091976;2013206645...,NaN
Wool,THEOS SWALLOW FORK RANCH LLP,2.081200e+09,Rio Blanco,ALL,"PO BOX 195, MEEKER, CO, 81641",THEOS SWALLOW FORK RANCH LLP; ID #: 0080200160...,0080200160;1088201167;9083205010;2081200207,"BANK OF THE SAN JUANS, DIVISION OF GLACIER BAN...",NONE,Sheep And Lamb,2003F115913 - 10/23/2003,2003F115913,10/23/2003,NaN,NaN,2003F115959 - 10/23/2003; 2008F077472 - 08/01/...,2003F115959;2008F077472;20182091976;2013206645...,NaN
Wool,"WAGLER, DEBRA",6.011232e+09,Douglas; El Paso; Larimer; Otero; Park; Pueblo,ALL,"2701 COUNTY ROAD 43, BAILEY, CO, 80421","WAGLER, DEBRA SUE; ID #: 6011232042",6011232042,UNITED STATES OF AMERICA ACTING THROUGH THE FA...,NONE,All Livestock; Sheep And Lamb,20242034554 - 04/18/2024,20242034554,04/18/2024,NaN,NaN,NaN,NaN,NaN


In [25]:
df['amendmentId'].value_counts()

Series([], Name: count, dtype: int64)

In [28]:
finO = open("/home/joe/bic_etl/cdos/business/business/data_source/masterlist.tsv",encoding="latin")
finT = open("/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv")

readerO = csv.DictReader(finO,delimiter="\t")
readerT = csv.DictReader(finT,delimiter="\t")

allO = []
allT = []
for row in readerO:
    allO.append(row)

for row in readerT:
    allT.append(row)


In [25]:
finO = open("/home/joe/bic_etl/bls/sm/data_transformed/sm.data.6.Colorado.tsv")
#finT = open("/home/joe/bic_etl/cdos/business/business/data_transformed/masterlist.tsv")


allO = []
allT = []
for row in finO:
    allO.append(row)
finO.close()
print(len(allO))

4852


In [23]:
hist={}
nhit=0
for nn,rowO in enumerate(allO):
    spl=rowO.split("\t")
    nn=len(spl)
    if nn not in hist:
        hist[nn]=0
    hist[nn]+=1

    # if nn == 8:
    #     print(rowO)
    #     print(spl)
    #     print()
    #     nhit+=1
    # if nhit > 10:
    #     break

In [24]:
hist

{11: 1412, 8: 3440}

In [34]:
finO = open("/home/joe/bic_etl/bls/sm/data_transformed/sm.data.6.Colorado.wages.tsv")
crap={}
for mm,line in enumerate(finO):
    if mm == 0:
        print(line)
    nn=len(line.split("\t"))
    if nn not in crap:
           crap[nn]=0
    crap[nn]+=1

year	month	area	superSector	industry	series	wklyHrsAll	hrlyEarningsAll	wklyEarningsAll	wklyHrsProd	hrlyEarningsProd	wklyEarningsProd



In [32]:
crap

{12: 4852}

In [18]:
hist={}
for row in allT:
    aid = row['amendmentId']
    if aid not in hist:
        hist[aid]=0
    hist[aid]+=1

In [ ]:
hist

In [ ]:
print(allO[0]['Additional Debtors'])
print("")
print(allT[0]['additionalDebtorId'])
print("")
print(allT[0]['additionalDebtors'])


In [ ]:
spl = allO[0]['Additional Debtors'].split(".")
addDebtors=""
for val in spl:
    indx=val.find("ID #")
    if indx > -1:
       db=val[indx+5:].strip()
       addDebtors+=f"{db},"
print(addDebtors)

In [ ]:
allT[0].keys()

In [ ]:
dfT

In [ ]:
a = set(dfO.columns) - set(dfT.columns)

In [ ]:
fin = open("/home/joe/school_programs.csv",quoting=3)
hist={}
for line in fin:
    spl = line.split(",") 
    nn=len(spl)
    if nn not in hist:
        hist[nn]=0
    hist[nn]+=1

In [ ]:
hist

In [ ]:
import csv
import datetime
from sodapy import Socrata


In [ ]:
file="/home/joe/bic_etl/cdos/business/nonprofit/data_source/reg_finan.tsv"
fin=open(file,encoding="latin")
tsum=0
hist={}
nlines=0
for line in fin:
    nlines+=1
    spl=line.split("\t") 
    for a in spl:
        if isinstance(a,str):
            if a.find(",") > -1:
               if a not in hist:
                  hist[a]=0
               hist[a]+=1
    # nn=len(spl)
    # if nn not in hist:
    #     hist[nn]=0
    # hist[nn]+=1
    # tsum+=len(line)
    # if nn != 35:
    #     print(nlines,len(line))
    #     print(line)
    #     for c in line:
    #         print(c,ord(c))

print("Total ",nlines)

In [ ]:
hist

In [ ]:
cimDatasets = []
bicHome = "/home/joe/bic_etl"
cim_url_query = 'data.colorado.gov'
with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
        if dataset['owner']['display_name'] == 'Colorado Information Marketplace':
            cimDatasets.append(dataset)

In [ ]:
cimDatasets[10]

In [ ]:
def getTransCols(title,datasets):
    tdict={}
    for dat in datasets:
        if dat["resource"]["name"] == title:
           break
    cols = dat["resource"]["columns_name"]
    desc = dat["resource"]["columns_description"]
    w4x4 = dat["resource"]["id"]
    typs = dat["resource"]['columns_datatype']
    for nn,col in enumerate(cols):
        tt = typs[nn]
        tdict[col]=tt
    return cols,desc,w4x4,typs,tdict

title="Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"
cols,desc,w4x4,dtypes,tdict = getTransCols(title,cimDatasets)

In [ ]:
tdict

In [ ]:
from dateutil.parser import parse

xrefs = {'str':'Text',
         'int':'Number',
         'float':'Number',
         'date':'Calendar date'
}



def is_date2(string, fuzzy=False):
    """
    Return whether the string can be interpreted as a date.

    :param string: str, string to check for date
    :param fuzzy: bool, ignore unknown tokens in string if True
    
    """
    fmts = ('%m/%d/%Y')
    try: 
        parse(string, fmts)
        return True

    except Exception as err:
        return False

def is_date(string, fmts):
    """
    Return whether the string can be interpreted as a date.

    :param string: str, string to check for date
    :param fuzzy: bool, ignore unknown tokens in string if True
    
    """
 
    try: 
        for fmt in fmts:
            datetime.datetime.strptime(string, fmt)
        return True

    except Exception as err:
        return False

file="/home/joe/bic_etl/cdos/business/nonprofit/data_source/reg_finan.tsv"
fin=open(file,encoding="latin")
reader = csv.DictReader(fin,delimiter="\t")
tsum=0
hist={}
histType={}
nlines=0
for line in reader:
    nlines+=1

    for c,val in line.items():
        t = type(val)
        if isinstance(val,str): 
            maybe=is_date(val,['%m/%d/%Y'])
        if maybe:
            t="date"
            if c == 'Os Exp Payaff':
                print(val)
        if c not in histType:
            histType[c]={}
        if t not in histType[c]:
            histType[c][t]=0
        histType[c][t]+=1
        if isinstance(val,str):
            if val.find(",") > -1:
               if c not in hist:
                  hist[c]=0
               hist[c]+=1

In [ ]:
for col,dct in histTyp.items():
    

In [ ]:
hist

In [ ]:
def getAsciiCodes():
    codesdf = pd.read_excel("/home/joe/Python/ascii-codes.xlsx",engine='openpyxl')
    asciiCodes={}
    for index,row in codesdf.iterrows():
        asciiCodes[row['DEC']] = {}
        asciiCodes[row['DEC']]['symbol'] = row['Symbol']
        asciiCodes[row['DEC']]['desc'] = row['Description']
    return asciiCodes
ascii=getAsciiCodes()

In [ ]:
ascii

In [ ]:
import csv

In [ ]:
file="/home/joe/bic_etl/cdos/business/nonprofit/data_source/persons_entity.tsv"
fin=open(file,encoding="latin")

lines=fin.readlines()
hist={}
# for line in lines:
#     # spl=line.split("\t")
#     # nn=len(spl)
#     for c in line: 
#       nn=ord(c)
#       if nn not in hist:
#         hist[nn]=0
#       hist[nn]+=1

In [ ]:
file="/home/joe/bic_etl/cdos/business/nonprofit/data_source/persons_entity.tsv"
dfN = pd.read_csv(file,delimiter="\t",encoding="latin",quoting=3)

file="/home/joe/bic_etl/cdos/business/nonprofit/data_transformed/persons_entity.tsv"
dfO = pd.read_csv(file,delimiter="\t",encoding="latin",quoting=3)

In [ ]:
dfC = pd.read_csv("/home/joe/work/JS2Python/data/CIM/Persons_Associated_with_Charitable_Organizations__Paid_Solicitors__and_Professional_Fundraising_Consultants_in_Colorado_-_FOR_TESTING_ONLY_20240923.csv")

In [ ]:
dfC.isna().sum()

In [ ]:
dfO.columns

In [ ]:
dfO.isna().sum()

In [ ]:
set(dfO.columns) - set(dfN.columns)

In [ ]:
df.columns

In [ ]:
len(lines)

In [ ]:
2965014*29


In [ ]:
for c,cnt in sorted(hist.items()):
    print(c,cnt,ascii[c]["desc"])

In [ ]:
len(lines[595444].split("\t"))

In [ ]:
fin=open(file,encoding="latin")

#reader = csv.reader(fin,delimiter="\t")
hist2={}
nline=0
try:
    for row in lines:
        reader=csv.reader(
        nline+=1
        nn=len(row)
        if nn not in hist2:
            hist2[nn]=0
        hist2[nn]+=1
except Exception as err:
    print("line: ",nline)
    print(err)

### CSV

In [ ]:
import io
#reader = csv.reader(lines,delimiter="\t")
hist={}
for nline,line in enumerate(lines):
   a=csv.reader(io.StringIO(line),delimiter="\t")
   b=next(a)
   nn=len(b)
   if nn   != 30:
       print(nline,nn,len(line))
       for mm,cc in enumerate(b):
           print("     ",mm,cc)
       print("---")
       print(line)
       lfs=0
       for mm,c in enumerate(lines[nline]):
          if ord(c) == 10:
              lfs+=1
          print(f'{mm:3d}  {ord(c):3d}  {c}  {ascii[ord(c)]["desc"]}')
       print(f"Total LIne Feeds: {lfs}")
       print("----------------")
   if nn not in hist:
       hist[nn]=0
   hist[nn]+=1

### a=306688
start=a-5
end=a+5

for nn in range(start,end+1):
    mm=lines[nn].split("\t")
    print(nn,len(mm),len(lines[nn]))
    print(lines[nn])
    lfs=0
    for c in lines[nn]:
        if ord(c) == 10:
            lfs+=1
        print(f'  {ord(c)}  {c}  {ascii[ord(c)]["desc"]}')
    print(f"Total LIne Feeds: {lfs}")
    print('---------------------------')

In [ ]:
hist

In [ ]:
lnum=[595443,685993,801309,1463135,2186072]
nums={}
for ln in [2186072]:
    hist={}
    start=ln-2
    end=ln+2
    for nn in range(start,end+1):
        line=lines[nn]
        nums[nn]=1
     #   hist[nn]={}
        for c in line:
            mm=ord(c)
            if mm not in hist:
                hist[mm]={}
            if nn not in hist[mm]:
                hist[mm][nn]=0
            hist[mm][nn]+=1

In [ ]:
print("CHR",end="")
for nn in nums:
    print(f" {nn:7d}",end="")
print("")
for c,dct in sorted(hist.items()):
    print(f"{c:3d}",end="")
 

    for nn in nums:
        if nn in dct:
            cnt=dct[nn]
        else:
            cnt=0
        print(f" {cnt:7d}",end="")
    print("")

In [ ]:
df=pd.read_csv(file,delimiter="\t",encoding="latin",on_bad_lines="warn")

In [ ]:
hist

In [ ]:

def decodeSIJ(file):
    fin = open(file,"r")
    
    infoSIJ = {}
    data = json.load(fin)
   
    fin.close()
    data['datasetID'],data['fileToPublish'],data['pathToSavedFile']
    infoSIJ['4x4']=data['datasetID']
    infoSIJ['file']=data['fileToPublish']
    infoSIJ['path']=data['pathToSavedFile']
    
    iFile = json.loads(data['controlFileContent'])
    for key,val in iFile.items():
        if 'columns' in val:
            inCols = val['columns']
    infoSIJ['inputColumns'] = inCols
    
    oFile=json.loads(data['ftpControlFileContent'])
    for key,val in oFile.items():
    #    display(key,val)
        if 'columns' in val:
            outCols = val['columns']
    infoSIJ['outputColumns'] = inCols
    return infoSIJ
    
infoSIJ = decodeSIJ("/home/joe/bic_etl/cdos/lobbyist/scripts/datasync/state_lobbyist_bills.sij")

In [ ]:
infoSIJ

In [ ]:
oFile=json.loads(data['ftpControlFileContent'])
for key,val in oFile.items():
#    display(key,val)
    if 'columns' in val:
        outCols = val['columns']
display(outCols)

In [ ]:
oFile['tsv']['columns']

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/business_entities.tsv",encoding="latin")
#fin = open("/home/joe/bic_etl/cdos/business/business/data_source/corpmstr.txt",encoding="latin")

#fout = open("/home/joe/bic_etl/cdos/business/business/data_source/business_entities_new.tsv","w")

hist={}
nlines=0
nout=0
nfields = {}
badlines=[]
for line in fin:
    spl = line.split("\t")
    nf = len(spl)
    nlines+=1
    if nf not in nfields:
        nfields[nf]=0
    nfields[nf]+=1
    bad=0
    for vals in spl:
        count=vals.count('"')
        if count != 2 and count != 0:
            display(count,vals)
            bad=1
            badlines.append(line)
        if count not in hist:
            hist[count]=0
        hist[count]+=1
    if bad == 0:
      nout+=1
 #     fout.write(line)
    else:
      print("Record: ",nlines)
      print(line)
display("# lines ",nlines)
display("# output lines ",nout)

In [ ]:
len(badlines)

In [ ]:
hist

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/business_entities.tsv",encoding="latin")

hist={}
nlines=0
nout=0
nfields = {}
bad=[]
for line in fin:
   
    spl = line.split("\t")
    nlines+=1
    if nlines==1:
        header=spl
    nn = len(spl)
    if nn not in hist:
        hist[nn]=0
    hist[nn]+=1
    if len(spl) != 37:
        bad.append(line)

In [ ]:
hist

In [ ]:
header.append(" ")
status=-1
for line in bad:
    print(line)
    spl=line.split("\t")
    if len(33333333333333333333333333333333333333333333333spl) > 37:
        for nn,c in enumerate(line):
            if ord(c) == 34:
                status*=-1
  #          print(ord(c),c,status)
            if  ord(c) == 9 and status==1:
               tmp=list(line)
               tmp[nn]=""
               line=''.join(tmp)
               print(ord(line[nn:nn+1]),line[nn:nn+1],len(line.split("\t")))
               c=""
        print(line) 
        print("------------------------------------")
        for nn,val in enumerate(line.split("\t")):
            print(nn,header[nn],val)
        #     if val.count('"') == 1:
        #         for c in val:
        #             print("   ",ord(c),c)
        

In [ ]:
fin = open("/home/joe/bic_etl_old/cdos/business/business/data_transformed/Business_Entities_in_Colorado.csv",encoding="latin")

hist={}
nlines=0
nout=0
nfields = {}
for line in fin:
    spl = line.split(",")
    nn = len(spl)
    if nn not in hist:
        hist[nn]=0
    hist[nn]+=1

In [ ]:
hist

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/business_entities.tsv",encoding="latin")
fout= open("/home/joe/bic_etl/cdos/business/business/data_source/business_entities-new.tsv","w")
nfixed=0
nlines=0
for line in fin:
    nlines+=1
    spl = line.split("\t")

    if len(spl) > 37:
        for nn,c in enumerate(line):
            if ord(c) == 34:
                status*=-1
  #          print(ord(c),c,status)
            if  ord(c) == 9 and status==1:
               tmp=list(line)
               tmp[nn]=""
               line=''.join(tmp)
               nfixed+=1
            
    fout.write(line)
    
display("Total Lines",nlines)
display("Lines Fixed",nfixed)

In [ ]:
#fin = open("/home/joe/bic_etl/cdos/business/business/data_source/business_entities.tsv",encoding="latin")
fin = open("/home/joe/bic_etl/cdos/business/business/data_source/corpmstr.txt",encoding="latin")

nlines=0
for line in fin:
    nlines+=1
    spl=line.split("\t")
    if len(spl) != 37:
       display(len(spl))
display("Total Lines",nlines)

In [ ]:
import chardet
file="/home/joe/bic_etl/cdos/business/business/data_source/corpmstr.txt"
rawdata = open(file, "rb").read()
encoding = chardet.detect(rawdata)['encoding']
print(encoding)

In [ ]:
!pip freeze

In [ ]:
file="/some/file/file.zip"

if file[-4:] == ".tsv":
    delim="\t"
elif file[-4:] == ".csv":
    delim=","
elif file[-5:] == ".xlsx" or file[-4:] == ".xls":
    delim="-1"
else:
    delim="N/A"
    

print(f"delimiter is :{delim}:")

In [ ]:
file="/home/joe/bic_etl/cdos/lobbyist/data_source/prof_bills.tsv"
fin = open(file,encoding="latin")

lines=fin.readlines()

for nline,line in enumerate(lines):
    spl = line.split("\t")
    if len(spl) != 16:
        print(nline,len(spl))
        print(line)

In [ ]:
df = pd.read_csv("/home/joe/bic_etl/cdos/business/nonprofit/data_source/persons_sol_ntcs.tsv",delimiter="\t")

In [ ]:
df.head()

In [ ]:
df["Sn Id"].value_counts()

In [ ]:
df = pd.read_csv("https://data.colorado.gov/resource/hyr8-d3v9.csv?$limit=999999999")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df2 = pd.read_csv("/home/joe/bic_etl/cdos/business/nonprofit/data_transformed/campaign_supervisors.csv")

In [ ]:
df2.head()

In [ ]:
df = pd.read_csv("/home/joe/bic_etl/cdos/business/nonprofit/data_transformed/reg_finan.tsv",delimiter="\t")

In [ ]:
df["unrestrictedAssests"].dtypes

In [ ]:
vals = df["unrestrictedAssests"].values.tolist()

In [ ]:
hist={}
nchecked=0
notchecked=0
for val in vals:
    if isinstance(val,str):
        for c in val:
            nn=ord(c)
            if nn not in hist:
                hist[nn]=0
            hist[nn]+=1
        nchecked+=1
    else:
        notchecked+=1
    

In [ ]:
for c,count in sorted(hist.items()):
    print(c,count)

In [ ]:
some = {}

In [ ]:
some[2024] = {}

In [ ]:
print(some)

In [ ]:
some[2024][10]={}

In [ ]:
print(some)

In [ ]:
some[2024][10][20]=0

In [ ]:
some

In [ ]:
invt = {}
# 
for index,row in df.iterrows():
    date = row["updt"]
    
    if yr not in invt:
        invt[yr]={}

    if mo not in invt[yr]:
        invt[yr][mo]={}

    if dy not in invt[yr][mo]:
        invt[yr][mo][dy]=0

    invt[yr][mo][dy]+=1
    
            

In [ ]:
invt

In [ ]:
invt[year][month][day] = count

In [ ]:
for year in [2023,2024]:
    for mo in range(1,13):
        print(f"{year:4d} {mo:2d}",end="")
        for dy in range(1,32):
            if year in invt and mo in invt[year] and dy in invt[year][mo]:
                cnt = invt[year][mo][dy]
            else:
                cnt=0
            print(f" {cnt:4d}",end="")
        print("")

In [ ]:
df1 = pd.read_csv("/home/joe/work/CDLE/cdle_0424_tables.csv")
df2 = pd.read_csv("/home/joe/work/CDLE/cdle_0701_tables.csv")


In [ ]:
set(df1['TABLE_NAME'].values.tolist()) - set(df2['TABLE_NAME'].values.tolist())

In [ ]:
set(df2['TABLE_NAME'].values.tolist()) - set(df1['TABLE_NAME'].values.tolist())

In [ ]:
fin = open("/home/joe/bic_etl/cdos/business/nonprofit/defs/sol_ntcs_ew9y-6tv9_src_trns_xrefs.json")
d = json.load(fin)

In [ ]:
d

In [ ]:
d.keys()

In [ ]:
df= pd.read_csv("/home/joe/bic_etl/cdos/business/nonprofit/data_source/sol_ntcs.txt",delimiter="\t",encoding="latin")

In [ ]:
df.columns

In [ ]:
dd=set(d['ew9y-6tv9'].keys())
da=set(df.columns)

In [ ]:
dd-da

In [ ]:
da-dd

In [ ]:
dd['ew9y-6tv9'].keys()

In [ ]:
inventory[seried_od][year][month]+=1

summary[series_id][199001,199012,200112]

In [ ]:
for series_id,lst in summary.items():
    print(series_id)
    for yrmo in sorted(lst):
        print("   ",yrmo)